# ConceptNet_lite para búsquedas de relaciones y recuperación de caminos

El propósito de esta notebook es experimentar las formas de búsqueda en el grafo de ConceptNet en ambas direcciones para cada palabra de EmoPro y encontrar una forma de recuperar el path de la búsqueda. 

In [1]:
# Importes
import pandas as pd
import conceptnet_lite
conceptnet_lite.connect("data/conceptnet.db", db_download_url=None)
from conceptnet_lite import Label, edges_for, edges_between

## Pendientes:

- Modificar las funciones de David para entender bien los métodos del grafo y hacer que las que no regresan nada regresen sets o listas
- Modificar las funciones que ya regresan un set para ver si pueden regresar una lista guardando el tipo de relación y el orden
- Trabajar en el algoritmo que recupera relaciones de una palabra en un sentido, relaciones de otra palabra en sentido opuesto y checar coincidencias
- Hacer pruebas en las cuatro combinaciones de sentidos ((->, <-), (<-, ->), (->, ->), (<-, <-))
- Pensar en un método de búsqueda en profundidad.
- Tratar con embeddings como heurística para priorizar búsqueda entre conceptos
- Buscar cómo paralelizar procesos 
- Probar una búsqueda recursiva con brethfirst con un número limitado de repeticiones 
- Asegurar se que el algoritmo regrese más de un camino si es que lo hay
- Probar hacer una clase que almacene el camino. Es probable que la función de búsqueda se a un método de esta clase



### Información general sobre conceptent_lite

In [2]:
%psource conceptnet_lite

from enum import Enum
from pathlib import Path
from typing import Iterable, Optional, Union

import peewee

from conceptnet_lite.db import CONCEPTNET_EDGE_COUNT, CONCEPTNET_DUMP_DOWNLOAD_URL, CONCEPTNET_DB_NAME
from conceptnet_lite.db import CONCEPTNET_DB_URL
from conceptnet_lite.db import Concept, Language, Label, Relation, RelationName, Edge
from conceptnet_lite.db import prepare_db, _open_db, _generate_db_path, download_db
from conceptnet_lite.utils import PathOrStr, _to_snake_case


def connect(
        db_path: PathOrStr = CONCEPTNET_DB_NAME,
        db_download_url: Optional[str] = CONCEPTNET_DB_URL,
        delete_compressed_db: bool = True,
        dump_download_url: str = CONCEPTNET_DUMP_DOWNLOAD_URL,
        load_dump_edge_count: int = CONCEPTNET_EDGE_COUNT,
        delete_compressed_dump: bool = True,
        delete_dump: bool = True,
) -> None:
    """Connect to ConceptNet database.

    This function connects to ConceptNet database. If it does not exists, there are two opt

In [3]:
dir(conceptnet_lite.Label)

['DoesNotExist',
 '__class__',
 '__data__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__isabstractmethod__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__rel__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__sql__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_coerce',
 '_meta',
 '_normalize_data',
 '_pk',
 '_pk_expr',
 '_populate_unsaved_relations',
 '_prune_fields',
 '_schema',
 'add_index',
 'alias',
 'bind',
 'bind_ctx',
 'bulk_create',
 'bulk_update',
 'clone',
 'coerce',
 'concepts',
 'copy',
 'create',
 'create_table',
 'delete',
 'delete_by_id',
 'delete_instance',
 'dependencies',
 'dirty_fields',
 'drop_table',
 'filter',
 'get',
 'get_by_id',
 'get_id',
 'get_or_create',
 'get_or_none',
 'id',
 'index',
 'insert',
 'insert_from',
 'insert_many',
 'is_alias',
 'is_dirty',

In [4]:
help(edges_between)

Help on function edges_between in module conceptnet_lite:

edges_between(start_concepts: Iterable[conceptnet_lite.db.Concept], end_concepts: Iterable[conceptnet_lite.db.Concept], relation: Union[conceptnet_lite.db.Relation, str, NoneType] = None, two_way: bool = False) -> peewee.ModelSelect



## Funciones para recuperar aristas del grafo

Las funciones son para recuperar conceptos que une dos conceptos dados, recuperación de aristas en una dirección y recuperación de todas las aristas

In [ ]:
# Relaciones con dirección para incorporar la información en la función de dirección
# No correr esta celda, sólo está aquí para entender los métodos de las aristas
"""
def relacion_direccion(w):
    s1=set()
    try:
        for e in edges_for(Label.get(text=w, language='en').concepts, same_language=True):
            if w== e.start.text and e.relation.name not in ["synonym","antonym","distinct_from","similar_to","related_to"]:
                print("direccion ->")
                print(e.start.text, "-", e.end.text, "|", e.relation.name,e)
                s1.add(e.end.text)
    except:
        pass
    return s1 """

In [2]:
# Todas las relaciones entre dos conceptos | es -> español | en -> inglés
def relaciones(wt,wh)-> list:
    l =[]
    concepts_wt = Label.get(text=wt, language='en').concepts
    concepts_wh = Label.get(text=wh, language='en').concepts
    for e in edges_between(concepts_wt, concepts_wh,two_way=True):
        l.append(e)
    return l

In [3]:
# Todas las relaciones entre dos conceptos, con dirección
def relaciones_generales(wt,wh) -> None:
    concepts_wt = Label.get(text=wt, language='en').concepts
    concepts_wh = Label.get(text=wh, language='en').concepts
    for e in edges_between(concepts_wt, concepts_wh,two_way=True):
        if wt== e.start.text:
            print(e.start.text, "-", e.end.text, "|", e.relation.name,e)

In [4]:
# Relación con dirección
def relacion_direccion(w)-> set:
    s1=set()
    try:
        for e in edges_for(Label.get(text=w, language='en').concepts, same_language=True):
            if w== e.start.text:
                #print("direccion ->")
                #print(e.start.text, "-", e.end.text, "|", e.relation.name,e)
                s1.add(e.end.text)
            elif w== e.end.text:
                s1.add(e.start.text)
    except:
        pass
    return s1

In [5]:
# Relación con dirección inversa
def relacion_direccion_inversa(w)-> set:
    s1=set()
    try:
        for e in edges_for(Label.get(text=w, language='en').concepts, same_language=True):
            if w== e.end.text:
                print("<-direccion")
                print(e.start.text, "-", e.end.text, "|", e.relation.name,e)
                s1.add(e.start.text)
    except:
        pass
    return s1

In [6]:
def relacion_word(w) -> None:
    try:
        for e in edges_for(Label.get(text=w, language='en').concepts, same_language=True):
            if w== e.end.text:
                print("derecha")
                print(e.start.text, "-", e.end.text, "|", e.relation.name,e)
            elif w== e.start.text:
                print("izquierda")
                print(e.start.text, "-", e.end.text, "|", e.relation.name,e)
    except:
        pass    

### Función recursiva para recperar caminos

In [10]:
# Pruebas de componentes de función recursiva porque entra en un loop que no debería

def buscarPrueba(word1:str, word2: str, runs = 5) -> dict:
    if runs == 0:
        print(0)
        return None
    rel = relaciones(word1, word2)
    if len(rel) != 0:
        print('sí hay relaciónes')
        print(rel)
        return rel
    else:
        print('no hubo relacions')
        return 0

In [ ]:
# Definición de función recursiva para buscar y regresar los paths entre palabras con máximo de 4 intentos
def busca_path(word1:str, word2:str, runs = 5)->dict:
    print(f'runs vale: {runs}')
    
    paths = {}
    r1 = relaciones(word1, word2)
    if len(r1) != 0:
        paths[word1] = r1
        paths[word2] = r1
        print(f'Hubo relaciones directas entre {word1}, {word2}')
        return paths
    if runs == 0:
            print(f'Límite de búsqueda, no se encontraron relaciones entre {word1}, {word2}')
            return None
        
    relaciones1 = relacion_direccion(word1)
    relaciones2 = relacion_direccion(word2)
    intersections = relaciones1.intersection(relaciones2)
    
    if len(intersections) != 0:
        print(f'Hay coincidencias en relaciones entre {word1}, {word2}, y son: {intersections}')
        print('guardando relaciones 1')
        paths[word1] = [relaciones(word1, i) for i in intersections] 
        print('guardando relaciones 2')
        paths[word2] = [relaciones(word2, i) for i in intersections]
        return paths
    else:
        for i in relaciones1:
            for j in relaciones2:
                print(f'No hay resultados, buscando de forma iterativa para {i}, {j}')
                
                paths[i, j] = busca_path(i,j, runs = runs-1)
    return paths


### Ejemplos

In [22]:
caminosDeLaVida = busca_path('paper', 'axe', runs = 2)

runs vale: 2
Hay coincidencias en relaciones entre paper, axe, y son: {'tree', 'trees', 'tool', 'rock', 'stock', 'rough', 'wood', 'slang'}
guardando relaciones 1
guardando relaciones 2


In [25]:
caminosDeLaVida['axe']

[[<Edge: /a/[/r/related_to/,/c/en/axe/,/c/en/tree/]>],
 [<Edge: /a/[/r/related_to/,/c/en/axe/,/c/en/trees/]>],
 [<Edge: /a/[/r/is_a/,/c/en/axe/,/c/en/tool/]>,
  <Edge: /a/[/r/related_to/,/c/en/axe/,/c/en/tool/]>,
  <Edge: /a/[/r/related_to/,/c/en/axe/n/wikt/en_1/,/c/en/tool/]>],
 [<Edge: /a/[/r/related_to/,/c/en/axe/n/wikt/en_1/,/c/en/rock/]>],
 [<Edge: /a/[/r/related_to/,/c/en/axe/n/wikt/en_1/,/c/en/stock/]>],
 [<Edge: /a/[/r/related_to/,/c/en/axe/v/wikt/en_1/,/c/en/rough/]>],
 [<Edge: /a/[/r/related_to/,/c/en/axe/,/c/en/wood/]>],
 [<Edge: /a/[/r/has_context/,/c/en/axe/n/wikt/en_1/,/c/en/slang/]>]]

In [ ]:
esp = relacion_direccion_inversa("nervousness" )

In [ ]:
esp

### Importe de palabras de EmoPro para hacer pruebas de búsqueda y recuperación de caminos

In [ ]:
rel1 = relacion_direccion_inversa('abandon')
rel2 = relacion_direccion_inversa('abandonment')

intersect = rel1.intersection(rel2)

intersect

In [ ]:
# incluir palabras en español y en inglés
with open('palabras_espaniol_ingles_1palabra.csv') as f:
    df = pd.read_csv(f)

In [ ]:
df

In [ ]:
words = {'esp':[w['Español'] for _,w in df.iterrows()], 'ing': [w['English'] for _,w in df.iterrows()]}

In [ ]:
word1 = relacion_direccion(words['ing'][3])

In [ ]:
word2 = relacion_direccion_inversa(words['ing'][45])

In [ ]:
inter = word1.intersection(word2)

In [ ]:
16*30

In [ ]:
relacion_word("nervioso")

In [ ]:
relacion_word("nervosidad")

### Importe del conjunto de datos de EmoPro para separar por afinidad con emoción discreta

In [ ]:
emopro= pd.read_csv('~/Documentos/MGP/Proyecto_tesis/Palabras_proto/EmoPro-Dataset.csv')
emopro['Word_Eng'] = df['English'] 

#División del ds por emoción discreta dominante
emopro_happy = emopro[emopro['Dominant_emotion']=='happiness']
emopro_sad = emopro[emopro['Dominant_emotion']=='sadness']
emopro_fear = emopro[emopro['Dominant_emotion']=='fear']
emopro_anger = emopro[emopro['Dominant_emotion']=='anger']
emopro_disgust = emopro[emopro['Dominant_emotion']=='disgust']

In [ ]:
# Diccionario para guardar y separ las palabras por emoción discreta y por grado de prototipicalidad
emociones = {'happiness': {'hap_highly_proto': [(row['Word'], row['Word_Eng']) for _, row in emopro_happy.iterrows() if row['Prototypicality_Mean']>=3], 
                'hap_low_proto': [(row['Word'], row['Word_Eng'] )for _, row in emopro_happy.iterrows() if row['Prototypicality_Mean']<3]},
            'sadeness': {'sad_highly_proto':[(row['Word'], row['Word_Eng']) for _, row in emopro_sad.iterrows() if row['Prototypicality_Mean']>=3],
                'sad_low_proto': [(row['Word'], row['Word_Eng']) for _, row in emopro_sad.iterrows() if row['Prototypicality_Mean']<3]},
            'anger': {'anger_highly_proto': [(row['Word'], row['Word_Eng']) for _, row in emopro_anger.iterrows() if row['Prototypicality_Mean']>=3],
                'anger_low_proto': [(row['Word'], row['Word_Eng']) for _, row in emopro_anger.iterrows() if row['Prototypicality_Mean']<3]},
            'fear':{'fear_highly_proto': [(row['Word'], row['Word_Eng']) for _, row in emopro_fear.iterrows() if row['Prototypicality_Mean']>=3],
                'fear_low_proto': [(row['Word'], row['Word_Eng']) for _, row in emopro_fear.iterrows() if row['Prototypicality_Mean']<3]},
            'disgust': {'disgust_highly_proto': [(row['Word'], row['Word_Eng']) for _, row in emopro_disgust.iterrows() if row['Prototypicality_Mean']>=3],
                'disgust_low_proto':[(row['Word'], row['Word_Eng']) for _, row in emopro_disgust.iterrows() if row['Prototypicality_Mean']<3]}}

# relaciones entre dos conceptos

In [ ]:
relaciones("try","look")

In [ ]:
relaciones("man","car")

In [ ]:
relaciones("side","full")

In [ ]:
relaciones("red","white")

In [ ]:
relacion_word("state")

In [ ]:
relacion_word("american")

# Diccionarios de relaciones

In [ ]:
#cargar relaciones para trabajar de manera local
df_diccionario = pd.read_pickle("data/Relaciones_generales.pickle")
df_diccionario_generales = df_diccionario.to_dict()

df_diccionario = pd.read_pickle("data/Relaciones_especificas.pickle")
df_diccionario_especificas = df_diccionario.to_dict()

In [ ]:
"sdasd" in df_diccionario_generales

In [ ]:
df_diccionario_generales["sdasd"]["is_a"]

In [ ]:
df_diccionario_especificas["man"]["is_a"]